In [37]:
import requests
import pandas as pd
import json
from io import StringIO
import numpy as np


In [26]:
PAGE_TITLE = "List_of_FIFA_World_Cup_official_match_balls"
API_URL = "https://en.wikipedia.org/w/api.php"

In [27]:
def get_page_html(title: str) -> str:
    """Pide a la MediaWiki API el HTML ya renderizado del artículo."""
    params = {
        "action": "parse",
        "page": title,
        "format": "json",
        "prop": "text",
    }
    resp = requests.get(API_URL, params=params, headers={
        "User-Agent": "WorldCupAnalyticsAgent/1.0 (portfolio project)"
    })
    resp.raise_for_status()
    data = resp.json()
    return data["parse"]["text"]["*"]


def extract_tables(html: str) -> list[pd.DataFrame]:
    """pandas.read_html parsea todas las <table> del HTML."""
    return pd.read_html(html)


In [28]:
html = get_page_html(PAGE_TITLE)

In [29]:
html

'<div class="mw-content-ltr mw-parser-output" lang="en" dir="ltr"><p class="mw-empty-elt">\n</p>\n<figure class="mw-default-size" typeof="mw:File/Thumb"><a href="/wiki/File:Giant_balls.jpg" class="mw-file-description"><img src="//upload.wikimedia.org/wikipedia/commons/thumb/f/fd/Giant_balls.jpg/250px-Giant_balls.jpg" decoding="async" width="250" height="167" class="mw-file-element" srcset="//upload.wikimedia.org/wikipedia/commons/thumb/f/fd/Giant_balls.jpg/500px-Giant_balls.jpg 2x" data-file-width="1690" data-file-height="1126" /></a><figcaption>Large inflatable replicas of Adidas World Cup balls in <a href="/wiki/Cape_Town" title="Cape Town">Cape Town</a>, 2009: from left to right, <a href="/wiki/Adidas_Telstar" title="Adidas Telstar">Telstar</a> (1970), <a href="/wiki/Telstar_Durlast" class="mw-redirect" title="Telstar Durlast">Telstar Durlast</a> (1974), <a href="/wiki/Adidas_Tango" title="Adidas Tango">Tango</a> (1978), <a href="/wiki/Tango_Espa%C3%B1a" class="mw-redirect" title="T

In [30]:
tables = pd.read_html(StringIO(html))

In [31]:
tables[0]

,Edition,Match ball,Match ball.1,Manufacturer,Panels,Additional information,Ref.
0,1930,Tiento (1st half) T-Model (2nd half),NaN,—,NaN,Two different balls were used in the final: Ar...,[3][4]
1,1934,Federale 102,NaN,ECAS (Ente Centrale Approvvigionamento Sportiv...,NaN,NaN,[5]
2,1938,Allen,NaN,"Allen, Paris",13,"Made up of leather, It had white cotton laces ...",[6]
3,1950,Duplo T,NaN,Superball,12,First ball to have no laces and introduce the ...,[7]
4,1954,Swiss World Champion,NaN,"Kost Sport, Basel",18,The first 18-panel ball.,[4][8]
5,1958,Top Star,NaN,"Sydsvenska Läder och Remfabriken, Ängelholm (a...",NaN,Chosen from 102 candidates in a blind test by ...,[9][10]
6,1962,Crack,NaN,"Señor Custodio Zamora H., San Miguel, Chile Re...",18,The Crack was the official ball. Referee Ken A...,[3][4][9][11]
7,1966,Challenge 4-Star,NaN,Slazenger,25,In orange or yellow. Selected in a blind test ...,[4][12]
8,1970,Telstar,NaN,Adidas,32 (20 hexagons and 12 pentagons),Telstar was the first 32-panel black-and-white...,[4][13]
9,1974,Telstar Durlast,NaN,Adidas,32 (20 hexagons and 12 pentagons),"The first polyurethane coated ball, making it ...",[4]


In [32]:
match_balls_df = tables[0].copy()
match_balls_df = match_balls_df[~match_balls_df['Edition'].str.contains('women')].reset_index(drop=True).copy()
match_balls_df

,Edition,Match ball,Match ball.1,Manufacturer,Panels,Additional information,Ref.
0,1930,Tiento (1st half) T-Model (2nd half),NaN,—,NaN,Two different balls were used in the final: Ar...,[3][4]
1,1934,Federale 102,NaN,ECAS (Ente Centrale Approvvigionamento Sportiv...,NaN,NaN,[5]
2,1938,Allen,NaN,"Allen, Paris",13,"Made up of leather, It had white cotton laces ...",[6]
3,1950,Duplo T,NaN,Superball,12,First ball to have no laces and introduce the ...,[7]
4,1954,Swiss World Champion,NaN,"Kost Sport, Basel",18,The first 18-panel ball.,[4][8]
5,1958,Top Star,NaN,"Sydsvenska Läder och Remfabriken, Ängelholm (a...",NaN,Chosen from 102 candidates in a blind test by ...,[9][10]
6,1962,Crack,NaN,"Señor Custodio Zamora H., San Miguel, Chile Re...",18,The Crack was the official ball. Referee Ken A...,[3][4][9][11]
7,1966,Challenge 4-Star,NaN,Slazenger,25,In orange or yellow. Selected in a blind test ...,[4][12]
8,1970,Telstar,NaN,Adidas,32 (20 hexagons and 12 pentagons),Telstar was the first 32-panel black-and-white...,[4][13]
9,1974,Telstar Durlast,NaN,Adidas,32 (20 hexagons and 12 pentagons),"The first polyurethane coated ball, making it ...",[4]


In [33]:
match_balls_df.rename(columns={
    'Edition': 'tournament_id',
    'Match ball': 'ball_name',
    'Manufacturer': 'manufacturer'}, inplace=True)

match_balls_df['tournament_id'] = 'WC-' + match_balls_df['tournament_id']
match_balls_df.drop(columns=['Match ball.1', 'Panels',	'Additional information', 'Ref.'], inplace=True)
match_balls_df

,tournament_id,ball_name,manufacturer
0,WC-1930,Tiento (1st half) T-Model (2nd half),—
1,WC-1934,Federale 102,ECAS (Ente Centrale Approvvigionamento Sportiv...
2,WC-1938,Allen,"Allen, Paris"
3,WC-1950,Duplo T,Superball
4,WC-1954,Swiss World Champion,"Kost Sport, Basel"
5,WC-1958,Top Star,"Sydsvenska Läder och Remfabriken, Ängelholm (a..."
6,WC-1962,Crack,"Señor Custodio Zamora H., San Miguel, Chile Re..."
7,WC-1966,Challenge 4-Star,Slazenger
8,WC-1970,Telstar,Adidas
9,WC-1974,Telstar Durlast,Adidas


In [38]:
# delete the first row
match_balls_df = match_balls_df.iloc[1:].reset_index(drop=True)

# add two personalized rows
tiento = {
    'tournament_id': 'WC-1930',
    'ball_name': 'Tiento',
    'manufacturer': np.nan
}
t_model = {
    'tournament_id': 'WC-1930',
    'ball_name': 'T-Model',
    'manufacturer': np.nan
}

match_balls_df = pd.concat([match_balls_df, pd.DataFrame([tiento, t_model])], ignore_index=True)
match_balls_df

,tournament_id,ball_name,manufacturer,ball_id
0,WC-1934,Federale 102,ECAS (Ente Centrale Approvvigionamento Sportiv...,2.0
1,WC-1938,Allen,"Allen, Paris",3.0
2,WC-1950,Duplo T,Superball,4.0
3,WC-1954,Swiss World Champion,"Kost Sport, Basel",5.0
4,WC-1958,Top Star,"Sydsvenska Läder och Remfabriken, Ängelholm (a...",6.0
5,WC-1962,Crack,"Señor Custodio Zamora H., San Miguel, Chile Re...",7.0
6,WC-1966,Challenge 4-Star,Slazenger,8.0
7,WC-1970,Telstar,Adidas,9.0
8,WC-1974,Telstar Durlast,Adidas,10.0
9,WC-1978,Tango,Adidas,11.0


In [40]:
# order by tournament_id
match_balls_df = match_balls_df.sort_values(by='tournament_id').reset_index(drop=True)
match_balls_df['ball_id'] = match_balls_df.index + 1
match_balls_df

,tournament_id,ball_name,manufacturer,ball_id
0,WC-1930,T-Model,NaN,1
1,WC-1930,Tiento,NaN,2
2,WC-1934,Federale 102,ECAS (Ente Centrale Approvvigionamento Sportiv...,3
3,WC-1938,Allen,"Allen, Paris",4
4,WC-1950,Duplo T,Superball,5
5,WC-1954,Swiss World Champion,"Kost Sport, Basel",6
6,WC-1958,Top Star,"Sydsvenska Läder och Remfabriken, Ängelholm (a...",7
7,WC-1962,Crack,"Señor Custodio Zamora H., San Miguel, Chile Re...",8
8,WC-1966,Challenge 4-Star,Slazenger,9
9,WC-1970,Telstar,Adidas,10


In [42]:
# edit the 1994 ball name
match_balls_df.loc[match_balls_df['tournament_id'] == 'WC-1994', 'ball_name'] = 'Questra'
# delete everything after the first comma in the manufacturer column for all rows
match_balls_df['manufacturer'] = match_balls_df['manufacturer'].str.split(',').str[0]
match_balls_df

,tournament_id,ball_name,manufacturer,ball_id
0,WC-1930,T-Model,NaN,1
1,WC-1930,Tiento,NaN,2
2,WC-1934,Federale 102,ECAS (Ente Centrale Approvvigionamento Sportivi),3
3,WC-1938,Allen,Allen,4
4,WC-1950,Duplo T,Superball,5
5,WC-1954,Swiss World Champion,Kost Sport,6
6,WC-1958,Top Star,Sydsvenska Läder och Remfabriken,7
7,WC-1962,Crack,Señor Custodio Zamora H.,8
8,WC-1966,Challenge 4-Star,Slazenger,9
9,WC-1970,Telstar,Adidas,10


In [43]:
match_balls_df.to_csv("data/processed/match_balls.csv", index=False)